# 🧠 06. Temporal Fusion Transformer (TFT) with PyTorch
**Capstone Project DS-02: Demand Forecasting & Inventory Optimization**

Implements deep multi-horizon forecasting with variable selection networks, temporal self-attention, and pinball multi-quantile loss ($P_{10}, P_{50}, P_{90}$).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data.data_loader import M5DataLoader
from src.features.build_features import FeaturePipeline
from src.models.tft_model import TFTForecaster
from src.models.metrics import evaluate_forecast

loader = M5DataLoader(data_dir='../data/sample')
df = loader.build_merged_dataset(max_days=120)
pipeline = FeaturePipeline(lags=[7, 14], rolling_windows=[7])
feat_df = pipeline.fit_transform(df)
train_df, val_df = loader.split_train_val(feat_df, horizon_days=28)

X_train, y_train = pipeline.get_feature_matrix(train_df)
X_val, y_val = pipeline.get_feature_matrix(val_df)


In [ ]:
# Fit TFT
tft = TFTForecaster(hidden_size=32, max_epochs=5, learning_rate=0.005)
tft.fit(X_train, y_train.values)
tft_preds = tft.predict(X_val)

tft_metrics = evaluate_forecast(y_val.values, tft_preds['point'], y_train.values, y_lower=tft_preds['q_0.10'], y_upper=tft_preds['q_0.90'])
print('TFT Validation Metrics:', tft_metrics)
